In [1]:
#
import os

# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os

    os.environ["MODIN_ENGINE"] = "ray"
    import ray

    ray.init(
        num_cpus=int(os.environ["MODIN_CPUS"]),
        runtime_env={"env_vars": {"__MODIN_AUTOIMPORT_PANDAS__": "1"}},
    )
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import numpy as np

# visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.figure_factory as ff
import plotly.graph_objects as go
import time

colors = ["gold", "mediumturquoise", "darkorange", "lightgreen"]
kaggle_colors = ["#39c5ff", "#ffffff", "#f2f2f2", "#9ca3a4", "#3f484b"]
font = "Rubik"

In [2]:
### cell 0 ###

df = pd.read_csv(
    "/home/dias-benchmarks/notebooks/madhurpant/beautiful-kaggle-2022-analysis/input/kaggle-survey-2022/kaggle_survey_2022_responses.csv"
)
factor = 10
df = pd.concat([df] * factor, ignore_index=True)
df.info()

/var/tmp/ipykernel_3409385/1269899871.py:3: DtypeWarning:

Columns (0,15,43,57,73,88,104,118,126,132,170,200,208,215,225,248,255,257,260,270,271,272,277,281,294) have mixed types. Specify dtype option on import or set low_memory=False.



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239980 entries, 0 to 239979
Columns: 296 entries, Duration (in seconds) to Q44_12
dtypes: object(296)
memory usage: 541.9+ MB


In [3]:
### cell 1 ###

df = df[1:]

In [7]:
### cell 2 ###

df["Duration (in seconds)"] = pd.to_numeric(
    df["Duration (in seconds)"], errors="coerce"
)

In [8]:
### cell 3 ###

df["Q3"].replace(
    ["Nonbinary", "Prefer not to say", "Prefer to self-describe"],
    "Others",
    inplace=True,
)

/var/tmp/ipykernel_3409385/1093918528.py:3: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [9]:
### cell 4 ###

gender_count = df.groupby(["Q3"]).size().reset_index().rename(columns={0: "count"})

In [10]:
### cell 5 ###

duration_count = (
    df.groupby(["Duration (in seconds)"])
    .size()
    .reset_index()
    .rename(columns={0: "count"})
)
duration_count = duration_count[:1000]

In [11]:
### cell 6 ###

age_count = df.groupby(["Q2"]).size().reset_index().rename(columns={0: "count"})

In [12]:
### cell 7 ###

# Compute top 10 non‐NA counts once and build the DataFrame directly
tmp = df["Q4"].value_counts(dropna=True).head(10)
country_count = pd.DataFrame({"Q4": tmp.index, "count": tmp.values})

In [13]:
### cell 8 ###

country_df = df.groupby(["Q4"]).size().reset_index().rename(columns={0: "count"})

In [14]:
### cell 9 ###

student_count = df.groupby(["Q5"]).size().reset_index().rename(columns={0: "count"})

In [15]:
### cell 10 ###

degree_count = df.groupby(["Q8"]).size().reset_index().rename(columns={0: "count"})
degree_count.loc[
    degree_count["Q8"]
    == "Some college/university study without earning a bachelor’s degree",
    "Q8",
] = "College Without Bachelors"

In [16]:
### cell 11 ###

experience_count = df.groupby(["Q16"]).size().reset_index().rename(columns={0: "count"})

In [17]:
### cell 12 ###

annual_income_df = df.groupby(["Q29"]).size().reset_index().rename(columns={0: "count"})

In [18]:
### cell 13 ###

gender_duration_count = (
    df.groupby(["Duration (in seconds)", "Q3"])
    .size()
    .reset_index()
    .rename(columns={0: "count"})
)
gender_duration_count = gender_duration_count[:3000]

In [19]:
### cell 14 ###

country_gender = (
    df.groupby(["Q4", "Q3"]).size().reset_index().rename(columns={0: "count"})
)

In [20]:
### cell 15 ###

age_duration_df = (
    df[df["Duration (in seconds)"] < 2000]
    .groupby("Q2")["Duration (in seconds)"]
    .mean()
    .round(1)
    .reset_index()
)

In [21]:
### cell 16 ###

education_gender = (
    df.groupby(["Q8", "Q3"]).size().reset_index().rename(columns={0: "count"})
)

In [22]:
### cell 17 ###

## Not Including India bcz it acts as an outlier here
top_9_countries = [
    "United States of America",
    "Other",
    "Brazil",
    "Nigeria",
    "Pakistan",
    "Japan",
    "China",
    "Egypt",
    "Mexico",
]

top_9_countries_df = df[df["Q4"].isin(top_9_countries)]

In [23]:
### cell 18 ###

gender_student = (
    df.groupby(["Q3", "Q5"]).size().reset_index().rename(columns={0: "count"})
)